In [190]:
%load_ext autoreload                                                                                                                               
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [191]:

import sys
import os
import numpy as np
import mujoco
import time
import mujoco_viewer

sys.path.append(os.path.abspath("../"))

from base_mujoco.utils import *
from base_mujoco.kinematics import *
from base_mujoco.slider_qt import *

In [192]:
model_path = "asset/example_scene_ffw_sh5.xml"
model_abs_path = os.path.abspath(model_path)


model = mujoco.MjModel.from_xml_path(model_abs_path)
data = mujoco.MjData(model)

body_names = get_body_names(model, data)
print(body_names)

['world', 'front_object_table', 'camera_ik', 'camera', 'camera2', 'camera3', 'base_link', 'left_wheel_steer', 'left_wheel_drive', 'right_wheel_steer', 'right_wheel_drive', 'rear_wheel_steer', 'rear_wheel_drive', 'lift_link', 'arm_base_link', 'head_link1', 'head_link2', 'arm_l_link1', 'arm_l_link2', 'arm_l_link3', 'arm_l_link4', 'arm_l_link5', 'arm_l_link6', 'arm_l_link7', 'hx5_l_base', 'finger_l_link1', 'finger_l_link2', 'finger_l_link3', 'finger_l_link4', 'finger_l_link5', 'finger_l_link6', 'finger_l_link7', 'finger_l_link8', 'finger_l_link9', 'finger_l_link10', 'finger_l_link11', 'finger_l_link12', 'finger_l_link13', 'finger_l_link14', 'finger_l_link15', 'finger_l_link16', 'finger_l_link17', 'finger_l_link18', 'finger_l_link19', 'finger_l_link20', 'arm_r_link1', 'arm_r_link2', 'arm_r_link3', 'arm_r_link4', 'arm_r_link5', 'arm_r_link6', 'arm_r_link7', 'hx5_r_base', 'finger_r_link1', 'finger_r_link2', 'finger_r_link3', 'finger_r_link4', 'finger_r_link5', 'finger_r_link6', 'finger_r_lin

In [193]:
print(data.qpos)

init_pose = {
    "arm_l_joint4": -1.55,
    "arm_r_joint4": -1.55,
    "finger_r_joint2": -1.5708,
    "finger_l_joint2": 1.5708,
    "finger_r_joint5":  0,   
    "finger_r_joint9":  0,   
    "finger_r_joint13": 0,   
    "finger_r_joint17": 0, 
    "finger_l_joint5":  0,  
    "finger_l_joint9":  0,   
    "finger_l_joint13": 0,   
    "finger_l_joint17": 0,   
}
for name, val in init_pose.items():
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)
    data.qpos[model.jnt_qposadr[jid]] = val

mujoco.mj_forward(model, data)
print(data.qpos)

[-0.25  0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.3  -0.3
  0.75  1.    0.    0.    0.  ]
[-0.25    0.      0.      1.      0.      0.      0.      0.      0.
  0.      0.      0.      0.      0.      0.      0.      0.      0.
  0.     -1.55    0.      0.      0.      0.      1.5708  0.      0.
  0.      0.      0.      0.      0.      0.      0.      0.      0.
  0.      0.      0.      0.      0.      0.      0.      0.      0.
  0.     -1.55    0.      0.      0.      0.     -1.5708  0.      0.
  0.      0.      0.      0.      0.      0.      0.      0.      0.
  0.      0.      0.      0.      0.      0.      0.      

In [194]:
cola_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'obj_coke')
cola_body_pos = data.xpos[cola_body_id]
print(cola_body_pos)

[ 0.3  -0.3   0.75]


In [195]:
""" Current POS ROT """
eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
eef_site_pos = data.site_xpos[eef_site_id]
eef_site_rmat = data.site_xmat[eef_site_id]
eef_site_rmat = eef_site_rmat.reshape(3, 3) 
print(f"Current EE position: {eef_site_pos}, Rotation matrix: {eef_site_rmat}")

""" Target POS ROT """
pos_target = np.array([0.3, -0.05, 0.8])
rmat_target = eef_site_rmat.copy()
print(f"Target EE position: {pos_target}, Rotation matrix: {rmat_target}")  

Current EE position: [ 0.28787684 -0.2175      1.12037556], Rotation matrix: [[ 0.02079483 -0.         -0.99978376]
 [ 0.          1.         -0.        ]
 [ 0.99978376  0.          0.02079483]]
Target EE position: [ 0.3  -0.05  0.8 ], Rotation matrix: [[ 0.02079483 -0.         -0.99978376]
 [ 0.          1.         -0.        ]
 [ 0.99978376  0.          0.02079483]]


In [196]:
""" CLIPPING ERROR """
POSITION_CLIPPING = 0.02
ROTATION_CLIPPING = 0.2

""" POS ROT RATIO """
POS_ROT_RATIO = 0.57

In [197]:
def get_ik_error_clipped(
        p_current,
        r_current,
        p_target,
        r_target,
        ):
    pos_error = p_target - p_current
    rmat_error = r_target @ r_current.T
    rotvec_error = rmat2rotvec(rmat_error)* POS_ROT_RATIO
    pos_error_clipped = np.clip(pos_error, -POSITION_CLIPPING, POSITION_CLIPPING)
    rotvec_error_clipped = np.clip(rotvec_error, -ROTATION_CLIPPING, ROTATION_CLIPPING)
    return pos_error_clipped, rotvec_error_clipped

In [198]:
model.nv
for i in range(model.njnt):
    print(model.jnt_type[i], model.joint(i).name)

0 floating_base
3 left_wheel_steer_joint
3 left_wheel_drive_joint
3 right_wheel_steer_joint
3 right_wheel_drive_joint
3 rear_wheel_steer_joint
3 rear_wheel_drive_joint
2 lift_joint
3 head_joint1
3 head_joint2
3 arm_l_joint1
3 arm_l_joint2
3 arm_l_joint3
3 arm_l_joint4
3 arm_l_joint5
3 arm_l_joint6
3 arm_l_joint7
3 finger_l_joint1
3 finger_l_joint2
3 finger_l_joint3
3 finger_l_joint4
3 finger_l_joint5
3 finger_l_joint6
3 finger_l_joint7
3 finger_l_joint8
3 finger_l_joint9
3 finger_l_joint10
3 finger_l_joint11
3 finger_l_joint12
3 finger_l_joint13
3 finger_l_joint14
3 finger_l_joint15
3 finger_l_joint16
3 finger_l_joint17
3 finger_l_joint18
3 finger_l_joint19
3 finger_l_joint20
3 arm_r_joint1
3 arm_r_joint2
3 arm_r_joint3
3 arm_r_joint4
3 arm_r_joint5
3 arm_r_joint6
3 arm_r_joint7
3 finger_r_joint1
3 finger_r_joint2
3 finger_r_joint3
3 finger_r_joint4
3 finger_r_joint5
3 finger_r_joint6
3 finger_r_joint7
3 finger_r_joint8
3 finger_r_joint9
3 finger_r_joint10
3 finger_r_joint11
3 finger_r

In [199]:
for i in range(model.njnt):
      if model.jnt_type[i] == 0:  # free joint만
          print(f"body: {model.body(model.jnt_bodyid[i]).name}, joint: {model.joint(i).name}")

body: base_link, joint: floating_base
body: obj_coke, joint: 


In [200]:
model.njnt

65

In [201]:
joints_use = [                                                                                                                                       
      "lift_joint",                                                                                                                                    
      "arm_r_joint1",                                                                                                                                  
      "arm_r_joint2",                                                                                                                                  
      "arm_r_joint3",                                                                                                                                  
      "arm_r_joint4",                                                                                                                                  
      "arm_r_joint5",
      "arm_r_joint6",
      "arm_r_joint7",
  ]

joints_use_qpos_idxs = [
      model.jnt_qposadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)]
      for name in joints_use
  ]
joints_use_qpos_idxs
def joint_names_to_act_ids(model, joint_names):                                                                                                                                                                                                        
  jnt_to_act = {}                                                                                                                                                                                                                                    
  for a in range(model.nu):                                                                                                                                                                                                                          
      j = model.actuator_trnid[a, 0]   # 이 actuator가 구동하는 joint id
      jnt_to_act[j] = a                                                 
  out = []                                                                                                                                                                                                                                           
  for n in joint_names:
      jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n)                                                                                                                                                                                   
      if jid == -1:                                               
          raise ValueError(f"joint '{n}' not found")                                                                                                                                                                                                 
      if jid not in jnt_to_act:                     
          raise ValueError(f"joint '{n}' has no actuator")                                                                                                                                                                                           
      out.append(jnt_to_act[jid])                                                                                                                                                                                                                    
  return out                     
                                                                                                                                                                                                                                                         
joints_use_ac_idxs = joint_names_to_act_ids(model, joints_use)


In [202]:
%gui qt
sliders = MultiSliderClass( # Slider for EE control
    n_slider      = 6,
    title         = 'Sliders for right arm Control',
    window_width  = 450,
    window_height = 300,
    x_offset      = 0,
    y_offset      = 100,
    slider_width  = 200,
    label_texts   = ['X','Y','Z','Roll-deg','Pitch-deg','Yaw-deg'],
    slider_mins   = [-1,-1,-1,-180,-180,-180],
    slider_maxs   = [+1,+1,+1,+180,+180,+180],
    slider_vals   = [0,0,0,0,0,0],
    resolutions   = [0.02,0.02,0.02,3.6,3.6,3.6], # range/50
    verbose       = False,
)

finger_sliders = MultiSliderClass( # Slider for EE control
    n_slider      = 2,
    title         = 'Sliders for right finger Control',
    window_width  = 450,
    window_height = 300,
    x_offset      = 0,
    y_offset      = 100,
    slider_width  = 200,
    label_texts   = ['right_thumb', 'right_grasp'],
    slider_mins   = [0,0],
    slider_maxs   = [+1,+1],
    slider_vals   = [0,0],
    resolutions   = [0.01,0.01], # range/50
    verbose       = False,
)

In [ ]:
# 오른손 finger 전체 (20개)
finger_r_joints = [f"finger_r_joint{i}" for i in range(1, 21)]
finger_r_ac_idxs = joint_names_to_act_ids(model, finger_r_joints)


# curl 진폭 (limit에서 약간 안쪽으로 잡아 충돌 여유 확보)
THUMB_PITCH_MAX = 1.4   # joint3
THUMB_IP_MAX    = 1.4   # joint4
FINGER_PIP_MAX  = 1.8   # joint 6,10,14,18
FINGER_DIP_MAX  = 1.4   # joint 7,11,15,19
FINGER_TIP_MAX  = 1.4   # joint 8,12,16,20

def make_finger_pose(thumb_s, grasp_s):
    """thumb_s, grasp_s in [0,1]. 0=open, 1=close. Returns {joint_name: qpos}."""
    s_t = float(np.clip(thumb_s, 0.0, 1.0))
    s_g = float(np.clip(grasp_s, 0.0, 1.0))
    pose = {}
    # Thumb
    pose["finger_r_joint3"] = THUMB_PITCH_MAX * s_t
    pose["finger_r_joint4"] = THUMB_IP_MAX    * s_t
    # 4 fingers
    for base in (5, 9, 13, 17):
        pose[f"finger_r_joint{base+1}"] = FINGER_PIP_MAX * s_g
        pose[f"finger_r_joint{base+2}"] = FINGER_DIP_MAX * s_g
        pose[f"finger_r_joint{base+3}"] = FINGER_TIP_MAX * s_g
    return pose

### IK with mj_step

In [204]:
model.body_gravcomp[:] = 1.0



In [205]:
hold_pose = data.qpos.copy()                                                                                                                       
                                                                                                                                                     
  # 매 프레임 (mj_step 전)
for a in range(model.nu):                                                                                                                          
    if a in joints_use_ac_idxs:
        continue                                                                                                                                   
    # 그 actuator가 구동하는 joint의 qpos를 현재 자세로 박기                                                                                       
    j = model.actuator_trnid[a, 0]                          
    qadr = model.jnt_qposadr[j]                                                                                                                    
    data.ctrl[a] = hold_pose[qadr]

In [ ]:
viewer = mujoco_viewer.MujocoViewer(model, data)
cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, "ikview")                                                                                                                                                
viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
viewer.cam.fixedcamid = cam_id


wrist_center_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'wrist_center')
p0 = data.site_xpos[wrist_center_id].copy()
eef_site_rmat = data.site_xmat[wrist_center_id]
R0 = eef_site_rmat.reshape(3, 3).copy()
q0 = data.qpos[joints_use_qpos_idxs]
q_ik_init = q0.copy()

while viewer.is_alive:
    sliders.update() # update slider
    xyzrpy = sliders.get_slider_values()

    qpos, error = solve_ik(
        model,
        data,
        joint_names=joints_use,
        eef_site_name="wrist_center", #"eef_site"
        q_init = q_ik_init,
        p_target=xyzrpy[:3] + p0,
        r_target=rpy_deg2r(xyzrpy[3:6])@R0,
        max_tick=500
    )
    
    if np.linalg.norm(error) < 1e-2: q_ik_init = qpos.copy()
    else: q_ik_init = q0.copy()

    data.ctrl[joints_use_ac_idxs] = qpos

    # finger

    finger_sliders.update()
    thumb_s, grasp_s = finger_sliders.get_slider_values()
    finger_pose = make_finger_pose(thumb_s, grasp_s)
    for jname, val in finger_pose.items():
        aid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, jname)
        data.ctrl[aid] = val

    for _ in range(10):
        mujoco.mj_step(model, data)
    
    if viewer.is_alive:
        viewer.add_marker(                                                                                                                                                                                                        
            pos=xyzrpy[:3] + p0,                                                                                                                                                                                                     
            size=np.array([0.01, 0.01, 0.01]),                                                                                                                                                                                 
            rgba=np.array([0.0, 1.0, 0.0, 0.5]),  # 초록색                                                                                                                                                              
            type=mujoco.mjtGeom.mjGEOM_SPHERE,                                                                                                                                                                                    
            label=""                                                                                                                                                                                                              
        )
        viewer.render()

viewer.close()

### IK with mj_forward

In [ ]:
# viewer = mujoco_viewer.MujocoViewer(model, data)
viewer = mujoco_viewer.MujocoViewer(model, data)
cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, "ikview")                                                                                                                                                
viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
viewer.cam.fixedcamid = cam_id

while viewer.is_alive:
    # get current pos, rotation
    eef_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'eef_site')
    eef_site_pos = data.site_xpos[eef_site_id]
    eef_site_rmat = data.site_xmat[eef_site_id]
    eef_site_rmat = eef_site_rmat.reshape(3, 3)

    pos_error, rotvec_error = get_ik_error_clipped(
        p_current=eef_site_pos,
        r_current=eef_site_rmat,
        p_target=pos_target,
        r_target=rmat_target
    )
    error = np.concatenate([pos_error, rotvec_error])

    # terminate condition
    if np.linalg.norm(pos_error) < 0.01 and np.linalg.norm(rotvec_error) < 0.01:
        print("Target reached!")

    
    jacobian_p, jacobian_r = get_jacobian(model, data, 'eef_site', type='site', joints_use=joints_use)

    jacobian = np.concat([jacobian_p, jacobian_r], axis=0)

    inversed_jacobian = get_pseudo_inverse(jacobian)

    qpos_error = inversed_jacobian @ error

    data.qpos[joints_use_qpos_idxs] += qpos_error
    mujoco.mj_forward(model, data)

    if viewer.is_alive:
        viewer.add_marker(                                                                                                                                                                                                        
            pos=pos_target,                                                                                                                                                                                                     
            size=np.array([0.01, 0.01, 0.01]),                                                                                                                                                                                 
            rgba=np.array([0.0, 1.0, 0.0, 0.5]),  # 초록색                                                                                                                                                              
            type=mujoco.mjtGeom.mjGEOM_SPHERE,                                                                                                                                                                                    
            label=""                                                                                                                                                                                                              
        )
        viewer.render()
        time.sleep(0.1)

viewer.close()

Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!
Target reached!


: 